# Valorant Round Prediction - Data Preprocessing Pipeline

Complete data preprocessing pipeline for two prediction models:
- **Model A** (Pre-Round): Predicts round winner before combat begins
- **Model B** (Live-Round): Predicts round winner during combat (after each kill event)

**Reads:** `pre_round.csv`, `live_round.csv`  
**Outputs:** Train/test CSVs in `./processed/` directory

**Model:** XGBoost (tree-based) -- no feature scaling needed

## Step 0: Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
# Configuration
DATA_DIR   = Path(".")
OUTPUT_DIR = DATA_DIR / "processed"

# Maps to remove (too few samples or different game mode)
RARE_MAPS = ["HURM_Alley", "HURM_Yard", "Pitt", "Canyon"]

# Outlier thresholds for live-round data
MAX_KILL_INDEX = 10   # Normal 5v5 round can have at most 10 kills
MAX_ATT_KILLS  = 5    # Max 5 enemies to kill
MAX_DEF_KILLS  = 5    # Max 5 attackers to kill

# Columns to drop (identifiers, not features)
ID_COLUMNS = ["source_file"]

# Train/test split ratio
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# Target column
TARGET = "winner"

print("Configuration loaded.")

## Step 1: Load Data

In [ ]:
pre_df  = pd.read_csv(DATA_DIR / "pre_round.csv")
live_df = pd.read_csv(DATA_DIR / "live_round.csv")

print(f"Pre-round:  {pre_df.shape[0]:,} rows x {pre_df.shape[1]} columns")
print(f"Live-round: {live_df.shape[0]:,} rows x {live_df.shape[1]} columns")

print("\nPre-round columns:", list(pre_df.columns))
print("Live-round columns:", list(live_df.columns))

In [ ]:
# Quick look at both datasets
print("=== Pre-Round (first 5 rows) ===")
pre_df.head()

In [ ]:
print("=== Live-Round (first 5 rows) ===")
live_df.head()

## Step 2: Drop NaN Winners

Rows where the `winner` column is NaN or empty have no label -- they can't be used for supervised learning.

In [ ]:
pre_before  = len(pre_df)
live_before = len(live_df)

# Handle both NaN and empty string cases
pre_df  = pre_df[pre_df[TARGET].notna() & (pre_df[TARGET] != "")].copy()
live_df = live_df[live_df[TARGET].notna() & (live_df[TARGET] != "")].copy()

print(f"Pre-round:  {pre_before:,} -> {len(pre_df):,} (removed {pre_before - len(pre_df):,})")
print(f"Live-round: {live_before:,} -> {len(live_df):,} (removed {live_before - len(live_df):,})")

## Step 3: Remove Rare/Invalid Maps

Maps with very few samples (Pitt, Canyon) or from a different game mode (HURM_*) are removed to avoid noise.

In [ ]:
pre_before  = len(pre_df)
live_before = len(live_df)

pre_df  = pre_df[~pre_df["map"].isin(RARE_MAPS)].copy()
live_df = live_df[~live_df["map"].isin(RARE_MAPS)].copy()

print(f"Removed maps: {RARE_MAPS}")
print(f"Pre-round:  {pre_before:,} -> {len(pre_df):,} (removed {pre_before - len(pre_df):,})")
print(f"Live-round: {live_before:,} -> {len(live_df):,} (removed {live_before - len(live_df):,})")
print(f"Remaining maps: {sorted(pre_df['map'].unique())}")

## Step 4: Remove Outliers (Live-Round Only)

Some match files have bugged data with `kill_index` up to 197 and `att_kills`/`def_kills` up to 100+.
In a normal 5v5 Valorant round, there can be at most 10 kills total. These anomalous rows are removed.

In [ ]:
live_before = len(live_df)

mask = (
    (live_df["kill_index"] <= MAX_KILL_INDEX) &
    (live_df["att_kills"]  <= MAX_ATT_KILLS) &
    (live_df["def_kills"]  <= MAX_DEF_KILLS)
)
live_df = live_df[mask].copy()

print(f"Thresholds: kill_index <= {MAX_KILL_INDEX}, att_kills <= {MAX_ATT_KILLS}, def_kills <= {MAX_DEF_KILLS}")
print(f"Live-round: {live_before:,} -> {len(live_df):,} (removed {live_before - len(live_df):,})")

## Step 5: Encode Target Variable

Convert `winner` from string to binary:
- `allies` = 1 (local team wins)
- `enemies` = 0 (opponent wins)

In [ ]:
mapping = {"allies": 1, "enemies": 0}

pre_df[TARGET]  = pre_df[TARGET].map(mapping)
live_df[TARGET] = live_df[TARGET].map(mapping)

print(f"Mapping: {mapping}")
print(f"Pre-round  class balance: {pre_df[TARGET].value_counts().to_dict()}")
print(f"Live-round class balance: {live_df[TARGET].value_counts().to_dict()}")

## Step 6: Encode Categorical Features

- `local_team_side`: Binary label encoding (attack=1, defense=0)
- `map`: Label encoding as integer IDs (alphabetical order)

> **Note:** These are categorical features encoded as integers for XGBoost.
> They are **NOT scaled** -- XGBoost is tree-based and handles integer-encoded
> categoricals natively. Scaling would destroy their meaning.

In [ ]:
# Binary encode local_team_side: attack=1, defense=0
side_map = {"attack": 1, "defense": 0}
pre_df["local_team_side"]  = pre_df["local_team_side"].map(side_map)
live_df["local_team_side"] = live_df["local_team_side"].map(side_map)

# Label encode map (alphabetical order for consistency)
all_maps = sorted(set(pre_df["map"].unique()) | set(live_df["map"].unique()))
map_to_int = {m: i for i, m in enumerate(all_maps)}

pre_df["map"]  = pre_df["map"].map(map_to_int)
live_df["map"] = live_df["map"].map(map_to_int)

print(f"local_team_side encoding: {side_map}")
print(f"map label encoding: {map_to_int}")

## Step 7: Feature Engineering

Create derived features that add predictive signal beyond the raw columns:

**Common features (both datasets):**
- `att_full_buy` / `def_full_buy` -- binary, team can afford full buy (>=19,500 credits)
- `att_eco` / `def_eco` -- binary, team is on eco round (<10,000 credits)
- `ult_adv` -- attacker ultimate advantage (att_ults_ready - def_ults_ready)

**Live-round only:**
- `alive_ratio` -- fraction of alive players on attacker side (0.0 to 1.0)
- `att_wiping` / `def_wiping` -- binary, one team has been fully eliminated
- `kill_progress` -- fraction of total kills so far in the round (0.0 to 1.0)

In [ ]:
# Economy thresholds (team total across 5 players)
FULL_BUY_THRESHOLD = 19500   # ~3900 per player (Vandal/Phantom + full armor + abilities)
ECO_THRESHOLD      = 10000   # ~2000 per player (pistol/eco round)

# -- Common features (pre-round + live-round) --
for df in [pre_df, live_df]:
    df["att_full_buy"] = (df["att_money"] >= FULL_BUY_THRESHOLD).astype(int)
    df["def_full_buy"] = (df["def_money"] >= FULL_BUY_THRESHOLD).astype(int)
    df["att_eco"]      = (df["att_money"] < ECO_THRESHOLD).astype(int)
    df["def_eco"]      = (df["def_money"] < ECO_THRESHOLD).astype(int)
    df["ult_adv"]      = df["att_ults_ready"] - df["def_ults_ready"]

# -- Live-round only features --
total_alive = live_df["att_alive"] + live_df["def_alive"]
live_df["alive_ratio"]   = live_df["att_alive"] / total_alive.replace(0, 1)  # avoid div/0
live_df["att_wiping"]    = (live_df["def_alive"] == 0).astype(int)
live_df["def_wiping"]    = (live_df["att_alive"] == 0).astype(int)
live_df["kill_progress"] = (live_df["att_kills"] + live_df["def_kills"]) / 10.0  # max 10 kills in 5v5

new_common = ["att_full_buy", "def_full_buy", "att_eco", "def_eco", "ult_adv"]
new_live   = ["alive_ratio", "att_wiping", "def_wiping", "kill_progress"]
print(f"Common features added: {new_common}")
print(f"Live-round features added: {new_live}")
print(f"Pre-round  total columns: {len(pre_df.columns)}")
print(f"Live-round total columns: {len(live_df.columns)}")

## Step 8: Drop ID Columns (Keep All Feature Columns)

Only drop `source_file` -- it's an identifier, not a feature.

We keep **both raw features AND diff features**:
- `att_money` + `def_money` + `economy_diff` (all kept -- the raw values carry extra signal beyond the diff)
- `score_won` + `score_lost` + `score_diff` (all kept)
- `att_alive` + `def_alive` + `alive_diff` (all kept, live-round only)
- `att_kills` + `def_kills` + `kill_diff` (all kept, live-round only)

> **Why keep both?** A diff of 0 at 5k vs 5k is very different from 25k vs 25k.
> The individual values provide context that the diff alone cannot capture.

`match_id` is saved separately for the grouped train/test split, then dropped.

In [ ]:
# Save match_ids before dropping (needed for grouped split)
pre_match_ids  = pre_df["match_id"].copy()
live_match_ids = live_df["match_id"].copy()

# Drop only identifier columns
pre_df  = pre_df.drop(columns=[c for c in ID_COLUMNS if c in pre_df.columns])
pre_df  = pre_df.drop(columns=["match_id"])
live_df = live_df.drop(columns=[c for c in ID_COLUMNS if c in live_df.columns])
live_df = live_df.drop(columns=["match_id"])

print(f"Dropped: {ID_COLUMNS + ['match_id']}")
print(f"\nPre-round  features ({len([c for c in pre_df.columns if c != TARGET])}): {[c for c in pre_df.columns if c != TARGET]}")
print(f"Live-round features ({len([c for c in live_df.columns if c != TARGET])}): {[c for c in live_df.columns if c != TARGET]}")

In [ ]:
# Verify the cleaned data
print("=== Pre-Round DataFrame ===")
pre_df.head()

In [ ]:
print("=== Live-Round DataFrame ===")
live_df.head()

## Step 9: Train/Test Split (Grouped by Match ID)

We group by `match_id` so that all rounds from the same match stay in the same split.
This prevents **data leakage** -- without this, the model could memorize match-specific patterns.

> **No feature scaling is applied.** XGBoost is tree-based and does not require
> normalized features. Scaling categorical features like `map` and `local_team_side`
> would actually be harmful.

In [ ]:
def train_test_split_grouped(df, match_ids, dataset_name):
    """
    Split data into train/test sets, ensuring all rounds from the same
    match stay in the same split (prevents data leakage).
    """
    splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(df, df[TARGET], groups=match_ids))

    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df  = df.iloc[test_idx].reset_index(drop=True)

    # Verify no match_id overlap
    train_matches = set(match_ids.iloc[train_idx])
    test_matches  = set(match_ids.iloc[test_idx])
    overlap = train_matches & test_matches

    print(f"  {dataset_name}:")
    print(f"    Train: {len(train_df):,} rows ({len(train_matches)} matches)")
    print(f"    Test:  {len(test_df):,} rows ({len(test_matches)} matches)")
    print(f"    Match overlap: {len(overlap)} (should be 0)")
    print(f"    Train class balance: {train_df[TARGET].value_counts().to_dict()}")
    print(f"    Test  class balance: {test_df[TARGET].value_counts().to_dict()}")

    return train_df, test_df

# Split both datasets
pre_train, pre_test   = train_test_split_grouped(pre_df, pre_match_ids, "Pre-Round")
live_train, live_test = train_test_split_grouped(live_df, live_match_ids, "Live-Round")

## Step 10: Save Processed Data

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pre_train.to_csv(OUTPUT_DIR / "pre_round_train.csv",   index=False)
pre_test.to_csv(OUTPUT_DIR / "pre_round_test.csv",     index=False)
live_train.to_csv(OUTPUT_DIR / "live_round_train.csv", index=False)
live_test.to_csv(OUTPUT_DIR / "live_round_test.csv",   index=False)

print(f"Saved to {OUTPUT_DIR}/")
print(f"  pre_round_train.csv   ({len(pre_train):,} rows)")
print(f"  pre_round_test.csv    ({len(pre_test):,} rows)")
print(f"  live_round_train.csv  ({len(live_train):,} rows)")
print(f"  live_round_test.csv   ({len(live_test):,} rows)")

## Validation

In [ ]:
all_dfs = {
    "pre_train":  pre_train,
    "pre_test":   pre_test,
    "live_train": live_train,
    "live_test":  live_test,
}

all_ok = True
for name, df in all_dfs.items():
    nan_count = df.isnull().sum().sum()
    if nan_count > 0:
        print(f"  [FAIL] {name} has {nan_count} NaN values!")
        all_ok = False
    else:
        print(f"  [OK] {name}: {df.shape[0]:,} rows x {df.shape[1]} cols, no NaN values")

if all_ok:
    print(f"\n  All validation checks passed!")
else:
    print(f"\n  Some validation checks failed!")

## Summary

In [ ]:
print("=" * 60)
print("PREPROCESSING COMPLETE!")
print("=" * 60)
print(f"\nPre-Round (Model A):")
print(f"  Features ({len([c for c in pre_train.columns if c != TARGET])}): {[c for c in pre_train.columns if c != TARGET]}")
print(f"  Train: {len(pre_train):,} rows | Test: {len(pre_test):,} rows")
print(f"\nLive-Round (Model B):")
print(f"  Features ({len([c for c in live_train.columns if c != TARGET])}): {[c for c in live_train.columns if c != TARGET]}")
print(f"  Train: {len(live_train):,} rows | Test: {len(live_test):,} rows")
print(f"\nNotes:")
print(f"  - No feature scaling applied (XGBoost is tree-based)")
print(f"  - Map and side are label-encoded integers (not scaled)")
print(f"  - Both raw features and diff features are kept")